## Housing Preprocessing

This notebook describes the preprocessing steps applied to the apartment sales data scraped from Booli.se.

The dataset contains the following features:

* **Date** – Date when the apartment was sold
* **Street** – Street address of the apartment
* **Selling price** – Final selling price of the apartment
* **Area-Municipality** – Area where the apartment is located and which municipality it belongs to.
* **Size_m2** – Living area in square meters
* **Rooms** – Number of rooms
* **Floor** – Floor on which the apartment is located
* **Price_per_m2** – Selling price per square meter
* **Price_development** – Change in value relative to the initial selling price

The scraping tool that collected the data does not always capture all feature values in their correct columns. In particular, missing values can cause subsequent values to shift into incorrect columns, resulting in an improperly structured dataset. The affected values were therefore manually reorganized into their correct columns using Excel.

The following sections describe the preprocessing steps applied to the CSV file generated from Excel.


In [1]:
from pathlib import Path

import pandas as pd

from utils.constants import DATA_PATH


In [2]:
HOUSING_DATA = DATA_PATH / "housing"

In [3]:
df = pd.read_csv(HOUSING_DATA / "apartment_sales.csv", sep=";")
df

,Date,Street,Selling price,Area-Municipality,Size_m2,Rooms,Floor,Price_per_m2,Price_development
0,2024-09-25,Sandhamnsgatan 30,6 310 000 kr,Lägenhet · Gärdet · Stockholm,64 m²,2 rum,vån 8,98 600 kr/m²,NaN
1,2024-09-25,Kungsklippan 11,2 950 000 kr,Lägenhet · Kungsholmen · Stockholm,30 m²,"1,5 rum",vån -1,98 300 kr/m²,NaN
2,2024-09-25,Smedsbacksgatan 14,2 530 000 kr,Lägenhet · Östermalm · Stockholm,26 m²,1 rum,NaN,97 300 kr/m²,NaN
3,2024-09-25,Sankt Göransgatan 98,6 500 000 kr,Lägenhet · Kungsholmen Fridhemsplan · Stockholm,67 m²,3 rum,vån 4,97 000 kr/m²,NaN
4,2024-09-25,Ringvägen 155A,2 600 000 kr,Lägenhet · Södermalm Katarina · Stockholm,27 m²,1 rum,vån 4,96 300 kr/m²,NaN
...,...,...,...,...,...,...,...,...,...
81032,2026-10-15,Mullbärsstigen 15,1 925 000 kr,Lägenhet · Tibble · Upplands-Bro,84 m²,3 rum,vån 2,22 900 kr/m²,"–3,5%"
81033,2026-10-15,Lädersättravägen 99,1 500 000 kr,Lägenhet · Kallhäll · Järfälla,62 m²,2 rum,vån 8,24 200 kr/m²,"–3,2%"
81034,2026-10-15,Prästgårdsgatan 2,8 400 000 kr,Lägenhet · Centrum · Sundbyberg,96 m²,4 rum,vån 5,87 500 kr/m²,NaN
81035,2026-10-15,Sickla Strand 11,3 650 000 kr,Lägenhet · Sickla · Nacka,49 m²,2 rum,vån 2,74 500 kr/m²,NaN


* Rename some columns
* Check if string columns have any weird labels for missing values such as "Adress saknas"
* Drop Weblink as they have no value

In [4]:
# Rename some feature names
df = df.rename(columns={
    "Date": "Selling_date",
    "Selling price": "Selling_price",
    "Size_m2": "Livingspace_m2",
    "Area-Municipality": "Area_Municipality",
    "Price_development": "Value_change"
})


In [5]:
df.columns.to_list()

['Selling_date',
 'Street',
 'Selling_price',
 'Area_Municipality',
 'Livingspace_m2',
 'Rooms',
 'Floor',
 'Price_per_m2',
 'Value_change']

In [6]:

# Check all columns for empty cells containing whitespace and the string "sakna"

for col in df.columns:
    sakna_count = df[col].str.contains(r'sakna', regex=True).sum()
    empty_count = df[col].str.contains(r'^\s*$', regex=True).sum() # ^\s*$: matches empty strings or strings with only
                                                                   # whitespace
    print(f"Column: {col}, Contains 'sakna' count: {sakna_count}, empty count: {empty_count}")
    if sakna_count > 0:
        df[col] = df[col].replace(r'sakna', pd.NA, regex=True)
    if empty_count > 0:
        df[col] = df[col].replace(r'^\s*$', pd.NA, regex=True)


Column: Selling_date, Contains 'sakna' count: 0, empty count: 0
Column: Street, Contains 'sakna' count: 10, empty count: 0
Column: Selling_price, Contains 'sakna' count: 0, empty count: 0
Column: Area_Municipality, Contains 'sakna' count: 0, empty count: 0
Column: Livingspace_m2, Contains 'sakna' count: 0, empty count: 0
Column: Rooms, Contains 'sakna' count: 0, empty count: 0
Column: Floor, Contains 'sakna' count: 0, empty count: 0
Column: Price_per_m2, Contains 'sakna' count: 0, empty count: 0
Column: Value_change, Contains 'sakna' count: 0, empty count: 0


* Remove unncessesary substrings in feature values
* Convert numerical features from strings to floats.
* Convert date to date object.

In [7]:
df.dtypes

Selling_date         str
Street               str
Selling_price        str
Area_Municipality    str
Livingspace_m2       str
Rooms                str
Floor                str
Price_per_m2         str
Value_change         str
dtype: object

In [8]:
replace_substrings_in_features = {
    "Selling_price": [("kr", "")],
    "Area_Municipality": [("Lägenhet · ", ""), (" · ", "_")],
    "Livingspace_m2": [("m²", ""), (",", ".")],
    "Rooms": [("rum", ""), (",", ".")],
    "Floor": [("vån", ""), ("BV", "0"), (",", ".")],
    "Price_per_m2": [("kr/m²", "")],
    "Value_change": [("+/-", ""), ("%", ""), ("+", ""), ("–", "-"), (",", ".")],
}

In [9]:
for feature, replacements in replace_substrings_in_features.items():
    for old, new in replacements:
        df[feature] = df[feature].str.replace(old, new, regex=False)
    df[feature] = df[feature].str.strip()
    df[feature] = df[feature].str.replace(r"\s+", "", regex=True)

In [10]:
df

,Selling_date,Street,Selling_price,Area_Municipality,Livingspace_m2,Rooms,Floor,Price_per_m2,Value_change
0,2024-09-25,Sandhamnsgatan 30,6310000,Gärdet_Stockholm,64,2,8,98600,NaN
1,2024-09-25,Kungsklippan 11,2950000,Kungsholmen_Stockholm,30,1.5,-1,98300,NaN
2,2024-09-25,Smedsbacksgatan 14,2530000,Östermalm_Stockholm,26,1,NaN,97300,NaN
3,2024-09-25,Sankt Göransgatan 98,6500000,KungsholmenFridhemsplan_Stockholm,67,3,4,97000,NaN
4,2024-09-25,Ringvägen 155A,2600000,SödermalmKatarina_Stockholm,27,1,4,96300,NaN
...,...,...,...,...,...,...,...,...,...
81032,2026-10-15,Mullbärsstigen 15,1925000,Tibble_Upplands-Bro,84,3,2,22900,-3.5
81033,2026-10-15,Lädersättravägen 99,1500000,Kallhäll_Järfälla,62,2,8,24200,-3.2
81034,2026-10-15,Prästgårdsgatan 2,8400000,Centrum_Sundbyberg,96,4,5,87500,NaN
81035,2026-10-15,Sickla Strand 11,3650000,Sickla_Nacka,49,2,2,74500,NaN


In [11]:
# Split Rooms values NUM+NUM
df[["Livingspace_m2", "Extra_livingspace_m2"]] = df["Livingspace_m2"].str.split("+", expand=True)
df["Extra_livingspace_m2"] = df["Extra_livingspace_m2"].fillna("0")
reorder_columns = ["Selling_date", "Street", "Selling_price", "Area_Municipality", "Livingspace_m2",
                   "Extra_livingspace_m2", "Rooms", "Floor", "Price_per_m2", "Value_change"]
df = df[reorder_columns]

In [12]:
numerical_features = ["Selling_price", "Livingspace_m2", "Extra_livingspace_m2",
                      "Rooms", "Floor", "Price_per_m2", "Value_change"]
for feature in numerical_features:
    df[feature] = pd.to_numeric(df[feature], errors='raise')

In [13]:
df["Value_change"] = df["Value_change"] * 0.01

In [14]:
df

,Selling_date,Street,Selling_price,Area_Municipality,Livingspace_m2,Extra_livingspace_m2,Rooms,Floor,Price_per_m2,Value_change
0,2024-09-25,Sandhamnsgatan 30,6310000,Gärdet_Stockholm,64.0,0.0,2.0,8.0,98600.0,NaN
1,2024-09-25,Kungsklippan 11,2950000,Kungsholmen_Stockholm,30.0,0.0,1.5,-1.0,98300.0,NaN
2,2024-09-25,Smedsbacksgatan 14,2530000,Östermalm_Stockholm,26.0,0.0,1.0,NaN,97300.0,NaN
3,2024-09-25,Sankt Göransgatan 98,6500000,KungsholmenFridhemsplan_Stockholm,67.0,0.0,3.0,4.0,97000.0,NaN
4,2024-09-25,Ringvägen 155A,2600000,SödermalmKatarina_Stockholm,27.0,0.0,1.0,4.0,96300.0,NaN
...,...,...,...,...,...,...,...,...,...,...
81032,2026-10-15,Mullbärsstigen 15,1925000,Tibble_Upplands-Bro,84.0,0.0,3.0,2.0,22900.0,-0.035
81033,2026-10-15,Lädersättravägen 99,1500000,Kallhäll_Järfälla,62.0,0.0,2.0,8.0,24200.0,-0.032
81034,2026-10-15,Prästgårdsgatan 2,8400000,Centrum_Sundbyberg,96.0,0.0,4.0,5.0,87500.0,NaN
81035,2026-10-15,Sickla Strand 11,3650000,Sickla_Nacka,49.0,0.0,2.0,2.0,74500.0,NaN


In [15]:
df.dtypes

Selling_date                str
Street                      str
Selling_price             int64
Area_Municipality           str
Livingspace_m2          float64
Extra_livingspace_m2    float64
Rooms                   float64
Floor                   float64
Price_per_m2            float64
Value_change            float64
dtype: object

* Investigate if Price_per_m2 can be dropped as this value can be derived from Selling_price and Livingspace_m2

In [16]:
# Select the empty living space entries and "Price_per_m2"
empty_livingspace_and_price_per_m2_df = df[df["Livingspace_m2"].isnull()][["Livingspace_m2", "Price_per_m2"]]
empty_price_per_m2_df = df["Price_per_m2"].isnull()

print(f"Empty Price_per_m2: {empty_price_per_m2_df.sum()}")
print(f"Empty Livingspace_m2:\n{empty_livingspace_and_price_per_m2_df.isnull().sum()}")

Empty Price_per_m2: 179
Empty Livingspace_m2:
Livingspace_m2    177
Price_per_m2      177
dtype: int64


The majority of livingspace values are present, meaning that Price_per_m2 can be derived. We just need to check that the derived approximately equals their corresponding values in the Price_per_m2 column.

In [17]:
difference = abs(df["Selling_price"] - df["Price_per_m2"] * df["Livingspace_m2"])
difference.describe().T

count    80858.000000
mean      1540.699387
std       1215.282541
min          0.000000
25%        600.000000
50%       1340.000000
75%       2200.000000
max      49700.000000
dtype: float64

Since maximum difference is around 50 000, it is justifiable to drop the column since selling prices are in the millions.

In [18]:
df.drop(columns=["Price_per_m2"], inplace=True)


In [19]:
df.head(10)

,Selling_date,Street,Selling_price,Area_Municipality,Livingspace_m2,Extra_livingspace_m2,Rooms,Floor,Value_change
0,2024-09-25,Sandhamnsgatan 30,6310000,Gärdet_Stockholm,64.0,0.0,2.0,8.0,NaN
1,2024-09-25,Kungsklippan 11,2950000,Kungsholmen_Stockholm,30.0,0.0,1.5,-1.0,NaN
2,2024-09-25,Smedsbacksgatan 14,2530000,Östermalm_Stockholm,26.0,0.0,1.0,NaN,NaN
3,2024-09-25,Sankt Göransgatan 98,6500000,KungsholmenFridhemsplan_Stockholm,67.0,0.0,3.0,4.0,NaN
4,2024-09-25,Ringvägen 155A,2600000,SödermalmKatarina_Stockholm,27.0,0.0,1.0,4.0,NaN
5,2024-09-25,Frejgatan 20,4400000,Vasastan_Stockholm,46.0,0.0,2.0,1.0,NaN
6,2024-09-25,Lustgårdsgatan 1,5950000,Kungsholmen_Stockholm,63.0,0.0,3.0,7.0,NaN
7,2024-09-25,Herkulesgatan 24,6125000,Norrmalm_Stockholm,65.5,0.0,3.0,6.0,NaN
8,2024-09-25,Responsgatan 18,4295000,Telefonplan_Stockholm,46.0,46.0,2.0,1.0,NaN
9,2024-09-25,Sankt Göransgatan 153,2285000,Stadshagen_Stockholm,24.5,0.0,1.0,5.0,NaN


We will now split Area_Municipality and filter municipalities that are not Stockholm, then map the areas to districts in Stockholm.

In [20]:
df[["Area", "Municipality"]] = df["Area_Municipality"].str.split("_", expand=True)
df.drop(columns=["Area_Municipality"], inplace=True)
reordered_columns = ["Selling_date", "Selling_price", "Street", "Area", "Municipality", "Livingspace_m2",
                     "Extra_livingspace_m2", "Rooms", "Floor", "Value_change"]
df = df[reordered_columns]

In [21]:
df

,Selling_date,Selling_price,Street,Area,Municipality,Livingspace_m2,Extra_livingspace_m2,Rooms,Floor,Value_change
0,2024-09-25,6310000,Sandhamnsgatan 30,Gärdet,Stockholm,64.0,0.0,2.0,8.0,NaN
1,2024-09-25,2950000,Kungsklippan 11,Kungsholmen,Stockholm,30.0,0.0,1.5,-1.0,NaN
2,2024-09-25,2530000,Smedsbacksgatan 14,Östermalm,Stockholm,26.0,0.0,1.0,NaN,NaN
3,2024-09-25,6500000,Sankt Göransgatan 98,KungsholmenFridhemsplan,Stockholm,67.0,0.0,3.0,4.0,NaN
4,2024-09-25,2600000,Ringvägen 155A,SödermalmKatarina,Stockholm,27.0,0.0,1.0,4.0,NaN
...,...,...,...,...,...,...,...,...,...,...
81032,2026-10-15,1925000,Mullbärsstigen 15,Tibble,Upplands-Bro,84.0,0.0,3.0,2.0,-0.035
81033,2026-10-15,1500000,Lädersättravägen 99,Kallhäll,Järfälla,62.0,0.0,2.0,8.0,-0.032
81034,2026-10-15,8400000,Prästgårdsgatan 2,Centrum,Sundbyberg,96.0,0.0,4.0,5.0,NaN
81035,2026-10-15,3650000,Sickla Strand 11,Sickla,Nacka,49.0,0.0,2.0,2.0,NaN


In [22]:
df = df[df["Municipality"] == "Stockholm"]
df.drop(columns=["Municipality"], inplace=True)

In [23]:
df

,Selling_date,Selling_price,Street,Area,Livingspace_m2,Extra_livingspace_m2,Rooms,Floor,Value_change
0,2024-09-25,6310000,Sandhamnsgatan 30,Gärdet,64.0,0.0,2.0,8.0,NaN
1,2024-09-25,2950000,Kungsklippan 11,Kungsholmen,30.0,0.0,1.5,-1.0,NaN
2,2024-09-25,2530000,Smedsbacksgatan 14,Östermalm,26.0,0.0,1.0,NaN,NaN
3,2024-09-25,6500000,Sankt Göransgatan 98,KungsholmenFridhemsplan,67.0,0.0,3.0,4.0,NaN
4,2024-09-25,2600000,Ringvägen 155A,SödermalmKatarina,27.0,0.0,1.0,4.0,NaN
...,...,...,...,...,...,...,...,...,...
81023,2026-10-15,3702000,Igeldammsgatan 10,Kungsholmen,35.0,0.0,1.5,2.0,0.124
81024,2026-10-15,3000000,Broddgränd 30,Västertorp,58.0,0.0,2.0,1.0,0.113
81025,2026-10-15,2235000,Stjärnsundsgatan 17,Ormkärr,67.0,0.0,3.0,3.0,0.018
81027,2026-10-15,7300000,Danderydsgatan 30A,Östermalm,41.0,0.0,2.0,2.0,0.000


In [24]:
districts = [
    "Bromma",
    "Enskede-Årsta-Vantör",
    "Farsta",
    "Hägersten-Älvsjö",
    "Hässelby-Vällingby",
    "Järva",
    "Kungsholmen",
    "Norra innerstaden",
    "Skarpnäck",
    "Skärholmen",
    "Södermalm",
]

In [25]:
import pandas as pd

# The 'Area' column contains the names of different areas in Stockholm.
# We create a dictionary that groups these areas into districts.
# ChatGPT generated this grouping based on the areas in Stockholm.
# We have not verified the accuracy of this grouping, but since this is a school project,
# it should be sufficient for our purposes.

district_groups = {
    "Norra innerstaden": """
        Gärdet; Östermalm; Vasastan; Norrmalm; Hjorthagen;
        ÖstermalmNedreGärdet; Birkastan; InvidKarlaplan;
        VasastanOdenplan; NorraDjurgårdsstaden; VasastanÖstermalm;
        NedreÖstermalm; VasastanVidTreLiljor; VasastanNorrmalm;
        VasastanBirkastan; NedreGärdet; ÖvreÖstermalm;
        VasastanHagastaden; Hagastaden; ÖstermalmKarlaplan;
        City; Ekhagen; Vasastaden; Sveakvarteren; Djurgården;
        VasastanNorrmalmÖstermalm; Vasastan-Birkastan; Odenplan;
        BirkastanVasastan; Ruddammen; NorraDjurgården;
        CentrumNorrmalmVasastan; Sibrien; VasastanSibirien;
        Karlaplan; Sibirien; VasastanOdenplanSanktEriksplan;
        RödaBergen; NorraDjurgårdstaden; VasastanRödabergen;
        VasastanSanktEriksplan; Lärkstan; VasastanAtlas;
        GärdetÖstemalm; Östermalmstorg; HagastadenVasastan;
        Atlas; Vasaparken; Norrtull; VasastanRödaBergen;
        Lärkstaden; Breitenfeldsgatan; ÖstermalmGärdet;
        ÖstermalmNorraDjurgården; Vasasatan;
        VasastanAtlasBirkastan; Centrum/Norrmalm/Vasastan;
        Rödabergen; CityNorrmalm;
        Gärdet/Ladugårdsgärdet/Djurgården; ÖstermalmDjurgården;
        NedreGärdetÖstermalm; ÖstermalmVasastan;
        NorrmalmStockholm; ÖstermalmLärkstaden;
        ÖstermalmNorrmalmVasastan; Rörstrand; Hagastan;
        GärdetÖstermalm; Atlasområdet; Östermalm/Gärdet/Hjorthagen;
        ÖstermalmHjorthagen; NorraDjurgårdenÖstermalm;
        Tegnerlunden; Östermalms; Strandvägen;
        Östermalm|Karlaplan; Vasastan|Norrmalm;
        NorraDjurgårdsstan; Vasastan/Norrmalm/Östermalm;
        Vasastan+Norrmalm; Starrängsringen;
        Östermalm/Norrmalm/City; ÖvreGärdet;
        Vasastan/Östermalm/Norrmalm; Tegnérlunden
    """,

    "Kungsholmen": """
        Kungsholmen; KungsholmenFridhemsplan; Stadshagen;
        LillaEssingen; NedreKungsholmen; StoraEssingen;
        Lindhagen; Kristineberg; Fredhäll; KungsholmenThorildsplan;
        KungsholmsStrand; Essingeöarna; Kungholmen;
        Kungsholmstorg; Fridhemsplan;
        KungsholmenKungsholmsStrand; Essingen; NorrMälarstrand;
        Thorildsplan; HornsbergsStrand; KungsholmenMarieberg;
        VästraKungsholmen; Kungshomstorg; KristinebergKungsholmen;
        KungsholmenLillaEssingen; Marieberg;
        KungsholmenStoraEssingen; Kronobergsparken;
        KungsholmenHornsbergsStrand; KungsholmenRådhuset;
        KungsholmenFredhäll; Rådhusparken; KungsholmenStadshagen;
        KungsholmenNedre; KungsholmenLindhagen; HornsbegrsStrand;
        LillaEssinge; KristinebergsStrand;
        KungsholmenNorrMälarstrand; KungsholmenKristineberg;
        Rådhuset; Hornsberg; KungsholmenSanktEriksområdet;
        KungsholmenEssingeöarna; StockholmKungsholmen;
        Thorildplan; StoraEssingen1; Hornsbergs
    """,

    "Södermalm": """
        SödermalmKatarina; Södermalm; HammarbySjöstad; Maria;
        SödermalmSofia; Högalid; Skanstull; Sofia; Katarina;
        SödermalmHögalid; Hornstull; Nytorget; GamlaStan;
        StockholmSödermalm; Reimersholme; SödraStation;
        Mosebacke; Sofo; SödermalmMaria; Mariaberget;
        Danviksklippan; SödermalmVintertullstorget; MariaMagdalena;
        Mariatorget; Maria/GamlaStan/Högalid; Söder; Zinkensdamm;
        SödermalmReimersholme; "Sofo"; SödermalmSofo;
        SödermalmSkanstullRingen; Tanto; Skinnarviksberget;
        SödermalmHornstull; NorraHammarbyhamnenSödermalm;
        Järntorget85; Södermalm–Zinkensdamm; Stockholm–Södermalm;
        BergsundsStrand; Martiatorget;
        Maria/GamlaStan/Högalid/Hornstull; SödermalmMosebacke;
        SödermalmMariatorget; Danvikstull;
        SödermalmSkinnarviksbergetHögalid; StockholmSofia;
        Södermalmgalid; Tantolunden
    """,

    "Bromma": """
        Bromma; Beckomberga; BrommaBlackeberg; Traneberg;
        BrommaÅkeshovBrommaplan; BrommaBällsta; Abrahamsberg;
        Riksby; Blackeberg; Ulvsunda; Mariehäll; BrommaNockeby;
        Åkeslund; Brommaplan; Minneberg; BrommaAbrahamsberg;
        Johannesfred; Ulsunda; Äppelviken; Annedal;
        BrommaTraneberg; BrommaBrommaplan; StoraMossen; Alvik;
        Åkeshov; Riskby; Nockeby; BrommaBeckomberga;
        BrommaStoraMossen; BrommaÄppelviken; NorraÄngby;
        BrommaRiksby; Ålsten; SödraÄngby; Smedslätten;
        BrommaNockebyhov; BrommaÅkeslund; BrommaKyrka;
        BrommaMinneberg; BrommaÅlsten; BrommaJohannesfred;
        BrommaAlvik; Nockebyhov; BeckombergaSöderbergaPark;
        BrommaMariehäll; BrommaÅkeshov; BrommaAnnedal;
        BrommaHöglandet; Eneby; Bällsta; BrommaUlvsunda;
        BrommaBrommaKyrka; BrommaSmedslätten; Höglandet;
        BrommaNorraÄngby; Olovslund
    """,

    "Hägersten-Älvsjö": """
        Telefonplan; Midsommarkransen; Aspudden; Hägerstensåsen;
        Gröndal; Nybodahöjden; Västertorp; Västberga; Hägersten;
        Liljeholmskajen; Hägersten-Liljeholmen; Älvsjö; Solberga;
        Fruängen; Älvsjöstaden; LångbroPark; Liljeholmen;
        Mälarhöjden; Hökmossen; Örnsberg; Hägerstenshamnen;
        ÄlvsjöCentrum; GamlaMidsommarkransen; ÄlvsjöSolberga;
        Västberga/Telefonplan/Midsommarkransen; Axelsberg;
        Ekensberg; Aspudden/Telefonplan/Hägerstensåsen;
        Långbro; LiljeholmenNybohov; GamlaÄlvsjö; GröndalStrand;
        Långsjö; Hägerstenåsen; Liseberg;
        Hägerstenshamnen/Örnsberg/Axelsberg;
        Solberga/Älvsjö/Hägersten; Nybohov; ÄlvsjöC;
        Aspudden/Midsommarkransen/Telefonplan; Telefoplan;
        Klubbacken; MidsommarkransenTelefonplan; Herrängen;
        VästbergaTelefonplanMidsommarkransen;
        Midsommarkransen/Telefonplan/Västberga; ÄlvsjöLiseberg
    """,

    "Enskede-Årsta-Vantör": """
        Årsta; Bandhagen; Enskede; Högdalen; Stureby; Hagsätra;
        Johanneshov; Svedmyra; Rågsved; Blåsut; GamlaEnskede;
        Ormkärr; Östbergahöjden; Östberga; Gullmarsplan;
        RågsvedBjursätrahöjden; Globen; ÖrbySlott;
        Enskede-Årsta-Vantör; EnskedeGård; Skärmarbrink; Örby;
        GamlaÖstberga; Älvsjöbadet; Snösätra; HagsätraÄlvsjöbadet;
        CentralaÅrsta; HagsätraOrmkärr; Kärringstan;
        Gamlaöstberga; Ensked; Skärmabrink; HagsätraBandhagen;
        Ormskärr; JohanneshovSkärmabrink; Årsa;
        Skärmarbrink/Johanneshov/Globen; Kärringsta´N
    """,

    "Farsta": """
        FarstaStrand; Farsta; Sköndal; Hökarängen;
        FarstaStadsdelsområde; Gubbängen; Farstastadsdelsområde;
        FarstaCentrum; Fagersjö; Tallkrogen; Larsboda;
        SköndalsCentrum; Enskede-Gubbängen; Klockelund
    """,

    "Skarpnäck": """
        Bagarmossen; Björkhagen; Kärrtorp; Skarpnäck;
        Enskededalen; SkarpnäcksGård; SkarpnäcksStadsdelsområde;
        Skarpnäcksstadsdelsområde; SkarpnäckKärrtorp; Skapnäck
    """,

    "Skärholmen": """
        Bredäng; Sätra; SkärholmensCentrum; Skärholmen;
        Vårberg; StockholmBredäng; Skärholmshöjden; Gransätra
    """,

    "Hässelby-Vällingby": """
        VällingbyCentrum; VällingbyRåcksta; HässelbyNorraVillastad;
        Vällingby; HässelbyStrand; Hässelby; VällingbyNälsta;
        HässelbyGård; Hässelby-Vällingby; Johannelund;
        HässelbySödraVillastad; Råcksta; HässelbyVillastad;
        VällingbyJohannelund; HässelbyLövstaAllé; VällingbyGrimsta;
        Nälsta; JohannelundsTrädgårdsstad; VällingbyParkstad;
        Vällingbyhöjden; HässelbyStand; SpångaNälsta; Grimsta;
        Vinsta; Kälvesta; VällingbyCentum
    """,

    "Järva": """
        Spånga; SpångaSundby; Sundby; Bromstensstaden; Kista;
        Tensta; Husby; KistaGård; SpångaCentrum; Akalla;
        SpångaSolhem; Solhem; Bromsten; SpångaBromsten;
        Rinkeby-Kista; AkallaTrädgårdsstaden; Akallahöjden;
        AkallaTrädgårdsstad; Kistahöjden; Spånga-Tensta;
        Ärvinge; Rinkebyterassen; KistaAkalla; KistaHusby;
        SpångaKyrka; Akalladalen; Rinkeby
    """,
}

# Convert the grouped representation into Area -> District.
area_to_district = {}

for district, text in district_groups.items():
    for area in text.split(";"):
        area = area.strip()

        if not area:
            continue

        if area in area_to_district:
            raise ValueError(f"Duplicate mapping for {area!r}")

        area_to_district[area] = district

# Some of the districts cannot be derived directly from Area.
# We use ChatGPT to map the missing districts based on the area and street information.
# Again, we have not verified the accuracy of these mappings, but we deem them valid enough
# for this school project.
mapped_districts = pd.read_csv(HOUSING_DATA / "mapped_districts.csv")

area_to_district_missing = dict(zip(mapped_districts["Area"], mapped_districts["District"]))

area_to_district.update(area_to_district_missing)

In [26]:
df["District"] = df["Area"].map(area_to_district)

In [27]:
print(df["District"].isna().sum())
df["District"].value_counts()

0


District
Norra innerstaden       11790
Södermalm                7802
Hägersten-Älvsjö         6668
Kungsholmen              6553
Enskede-Årsta-Vantör     4658
Bromma                   4197
Skarpnäck                2016
Farsta                   1325
Hässelby-Vällingby       1077
Järva                     976
Skärholmen                387
Name: count, dtype: int64

In [28]:
df

,Selling_date,Selling_price,Street,Area,Livingspace_m2,Extra_livingspace_m2,Rooms,Floor,Value_change,District
0,2024-09-25,6310000,Sandhamnsgatan 30,Gärdet,64.0,0.0,2.0,8.0,NaN,Norra innerstaden
1,2024-09-25,2950000,Kungsklippan 11,Kungsholmen,30.0,0.0,1.5,-1.0,NaN,Kungsholmen
2,2024-09-25,2530000,Smedsbacksgatan 14,Östermalm,26.0,0.0,1.0,NaN,NaN,Norra innerstaden
3,2024-09-25,6500000,Sankt Göransgatan 98,KungsholmenFridhemsplan,67.0,0.0,3.0,4.0,NaN,Kungsholmen
4,2024-09-25,2600000,Ringvägen 155A,SödermalmKatarina,27.0,0.0,1.0,4.0,NaN,Södermalm
...,...,...,...,...,...,...,...,...,...,...
81023,2026-10-15,3702000,Igeldammsgatan 10,Kungsholmen,35.0,0.0,1.5,2.0,0.124,Kungsholmen
81024,2026-10-15,3000000,Broddgränd 30,Västertorp,58.0,0.0,2.0,1.0,0.113,Hägersten-Älvsjö
81025,2026-10-15,2235000,Stjärnsundsgatan 17,Ormkärr,67.0,0.0,3.0,3.0,0.018,Enskede-Årsta-Vantör
81027,2026-10-15,7300000,Danderydsgatan 30A,Östermalm,41.0,0.0,2.0,2.0,0.000,Norra innerstaden


In [29]:
# We now drop the Area column as it is no longer needed
df = df.drop(columns=["Area"])
reorder_columns = ["Selling_date", "Selling_price", "Street", "District", "Livingspace_m2", "Extra_livingspace_m2",
                   "Rooms", "Floor", "Value_change"]
df = df[reorder_columns]

In [30]:
df

,Selling_date,Selling_price,Street,District,Livingspace_m2,Extra_livingspace_m2,Rooms,Floor,Value_change
0,2024-09-25,6310000,Sandhamnsgatan 30,Norra innerstaden,64.0,0.0,2.0,8.0,NaN
1,2024-09-25,2950000,Kungsklippan 11,Kungsholmen,30.0,0.0,1.5,-1.0,NaN
2,2024-09-25,2530000,Smedsbacksgatan 14,Norra innerstaden,26.0,0.0,1.0,NaN,NaN
3,2024-09-25,6500000,Sankt Göransgatan 98,Kungsholmen,67.0,0.0,3.0,4.0,NaN
4,2024-09-25,2600000,Ringvägen 155A,Södermalm,27.0,0.0,1.0,4.0,NaN
...,...,...,...,...,...,...,...,...,...
81023,2026-10-15,3702000,Igeldammsgatan 10,Kungsholmen,35.0,0.0,1.5,2.0,0.124
81024,2026-10-15,3000000,Broddgränd 30,Hägersten-Älvsjö,58.0,0.0,2.0,1.0,0.113
81025,2026-10-15,2235000,Stjärnsundsgatan 17,Enskede-Årsta-Vantör,67.0,0.0,3.0,3.0,0.018
81027,2026-10-15,7300000,Danderydsgatan 30A,Norra innerstaden,41.0,0.0,2.0,2.0,0.000


In [32]:
df.to_csv(HOUSING_DATA / "processed" / "processed_apartment_data.csv", index=False)